In [4]:

from collections import deque
import heapq
import time

# ---------------------------------------------------------
# 1. GRAPH INPUT
# ---------------------------------------------------------

def read_graph():
    graph = {}
    heuristic = {}

    n = int(input("Enter number of nodes: "))

    print("Enter node names separated by spaces (example: A B C D):")
    nodes = input().upper().split()

    if len(nodes) != n:
        print("Number of node names must match the number of nodes.")
        return None, None, None

    for node in nodes:
        graph[node] = []

    m = int(input("Enter number of edges: "))

    print("Enter each edge as: source destination cost")
    print("Example: A B 2")

    for _ in range(m):
        u, v, cost = input().upper().split()
        cost = float(cost)

        if u not in graph or v not in graph:
            print("Unknown node entered.")
            return None, None, None

        graph[u].append((v, cost))
        graph[v].append((u, cost))

    start = input("Enter start node: ").upper()
    goal = input("Enter goal node: ").upper()

    if start not in graph or goal not in graph:
        print("Start or goal node is not in the graph.")
        return None, None, None

    return graph, start, goal


def read_heuristics(graph, goal):
    heuristic = {}

    print("Enter heuristic value for each node (estimated distance to goal).")

    for node in graph:
        heuristic[node] = float(input(f"h({node}) = "))

    return heuristic


def path_cost(graph, path):
    total = 0

    for i in range(len(path) - 1):
        for neighbor, cost in graph[path[i]]:
            if neighbor == path[i + 1]:
                total += cost
                break

    return total


def show_result(name, path, expanded, start_time, graph=None):
    elapsed = (time.perf_counter() - start_time) * 1000

    print(f"\n{name} Result")

    if path:
        print("Path:", " -> ".join(path))

        if graph is not None:
            print("Path cost:", path_cost(graph, path))
        else:
            print("Path length (edges):", len(path) - 1)

    else:
        print("No path found.")

    print("Nodes expanded:", expanded)
    print(f"Execution time: {elapsed:.3f} ms")


# ---------------------------------------------------------
# 2. UNINFORMED SEARCH: BFS
# ---------------------------------------------------------

def bfs(graph, start, goal):
    start_time = time.perf_counter()

    queue = deque([[start]])
    visited = {start}
    expanded = 0

    while queue:
        path = queue.popleft()
        node = path[-1]
        expanded += 1

        if node == goal:
            show_result("Breadth-First Search (BFS)", path, expanded, start_time)
            return

        for neighbor, _ in graph[node]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(path + [neighbor])

    show_result("Breadth-First Search (BFS)", None, expanded, start_time)


# ---------------------------------------------------------
# 3. UNINFORMED SEARCH: DFS
# ---------------------------------------------------------

def dfs(graph, start, goal):
    start_time = time.perf_counter()

    stack = [[start]]
    visited = set()
    expanded = 0

    while stack:
        path = stack.pop()
        node = path[-1]

        if node in visited:
            continue

        visited.add(node)
        expanded += 1

        if node == goal:
            show_result("Depth-First Search (DFS)", path, expanded, start_time)
            return

        for neighbor, _ in reversed(graph[node]):
            if neighbor not in visited:
                stack.append(path + [neighbor])

    show_result("Depth-First Search (DFS)", None, expanded, start_time)


# ---------------------------------------------------------
# 4. INFORMED SEARCH: GREEDY BEST-FIRST SEARCH
# ---------------------------------------------------------

def greedy_best_first(graph, start, goal, heuristic):
    start_time = time.perf_counter()

    frontier = [(heuristic[start], [start])]
    visited = set()
    expanded = 0

    while frontier:
        _, path = heapq.heappop(frontier)
        node = path[-1]

        if node in visited:
            continue

        visited.add(node)
        expanded += 1

        if node == goal:
            show_result("Greedy Best-First Search", path, expanded, start_time, graph)
            return

        for neighbor, _ in graph[node]:
            if neighbor not in visited:
                heapq.heappush(
                    frontier, (heuristic[neighbor], path + [neighbor])
                )

    show_result("Greedy Best-First Search", None, expanded, start_time, graph)


# ---------------------------------------------------------
# 5. INFORMED SEARCH: A*
# ---------------------------------------------------------

def a_star(graph, start, goal, heuristic):
    start_time = time.perf_counter()

    frontier = [(heuristic[start], 0, start, [start])]
    best_cost = {start: 0}
    expanded = 0

    while frontier:
        f_cost, g_cost, node, path = heapq.heappop(frontier)

        if g_cost != best_cost.get(node):
            continue

        expanded += 1

        if node == goal:
            show_result("A* Search", path, expanded, start_time, graph)
            return

        for neighbor, edge_cost in graph[node]:
            new_cost = g_cost + edge_cost

            if new_cost < best_cost.get(neighbor, float("inf")):
                best_cost[neighbor] = new_cost

                heapq.heappush(
                    frontier,
                    (new_cost + heuristic[neighbor], new_cost,
                     neighbor, path + [neighbor])
                )

    show_result("A* Search", None, expanded, start_time, graph)


# ---------------------------------------------------------
# 6. LOCAL SEARCH: HILL CLIMBING
# ---------------------------------------------------------

def hill_climbing():
    print("\nHill Climbing maximizes a user-provided list of values.")

    values = list(map(float, input(
        "Enter values separated by spaces (example: 1 4 7 5 3): "
    ).split()))

    if not values:
        print("No values entered.")
        return

    start_time = time.perf_counter()

    current = 0
    steps = 0

    while True:
        best = current

        if current > 0 and values[current - 1] > values[best]:
            best = current - 1

        if current < len(values) - 1 and values[current + 1] > values[best]:
            best = current + 1

        if best == current:
            break

        current = best
        steps += 1

    elapsed = (time.perf_counter() - start_time) * 1000

    print("\nHill Climbing Result")
    print("Best value found:", values[current])
    print("Position (0-based index):", current)
    print("Moves made:", steps)
    print(f"Execution time: {elapsed:.3f} ms")
    print("Note: Hill climbing may stop at a local maximum.")


# ---------------------------------------------------------
# 7. CONSTRAINT SATISFACTION: MAP COLORING
# ---------------------------------------------------------

def map_coloring():
    print("\nMap Coloring using Backtracking")

    regions = input("Enter region names separated by spaces: ").upper().split()
    colors = input("Enter available colors separated by spaces: ").upper().split()

    if not regions or not colors:
        print("Regions and colors cannot be empty.")
        return

    graph = {region: set() for region in regions}

    edge_count = int(input("Enter number of neighboring region pairs: "))

    print("Enter each neighboring pair (example: A B)")

    for _ in range(edge_count):
        a, b = input().upper().split()

        if a not in graph or b not in graph:
            print("Unknown region entered.")
            return

        graph[a].add(b)
        graph[b].add(a)

    assignment = {}
    nodes_tested = 0

    def is_safe(region, color):
        for neighbor in graph[region]:
            if assignment.get(neighbor) == color:
                return False

        return True

    def backtrack(index):
        nonlocal nodes_tested

        if index == len(regions):
            return True

        region = regions[index]

        for color in colors:
            nodes_tested += 1

            if is_safe(region, color):
                assignment[region] = color

                if backtrack(index + 1):
                    return True

                del assignment[region]

        return False

    start_time = time.perf_counter()

    solved = backtrack(0)

    elapsed = (time.perf_counter() - start_time) * 1000

    print("\nMap Coloring Result")

    if solved:
        for region in regions:
            print(f"{region} -> {assignment[region]}")
    else:
        print("No valid coloring exists with the given colors.")

    print("Color assignments tested:", nodes_tested)
    print(f"Execution time: {elapsed:.3f} ms")


# ---------------------------------------------------------
# MAIN MENU
# ---------------------------------------------------------

def main():
    while True:

        print("\n========== AI SEARCH ALGORITHMS ==========")
        print("1. Breadth-First Search (BFS)")
        print("2. Depth-First Search (DFS)")
        print("3. Greedy Best-First Search")
        print("4. A* Search")
        print("5. Hill Climbing")
        print("6. Map Coloring (Backtracking CSP)")
        print("7. Exit")

        choice = input("Enter your choice (1-7): ").strip()

        if choice in ("1", "2", "3", "4"):

            graph, start, goal = read_graph()

            if graph is None:
                continue

            if choice == "1":
                bfs(graph, start, goal)

            elif choice == "2":
                dfs(graph, start, goal)

            else:
                heuristic = read_heuristics(graph, goal)

                if choice == "3":
                    greedy_best_first(graph, start, goal, heuristic)

                else:
                    a_star(graph, start, goal, heuristic)

        elif choice == "5":
            hill_climbing()

        elif choice == "6":
            map_coloring()

        elif choice == "7":
            print("Program ended.")
            break

        else:
            print("Invalid choice. Please enter a number from 1 to 7.")


if __name__ == "__main__":
    main()



========== AI SEARCH ALGORITHMS ==========
1. Breadth-First Search (BFS)
2. Depth-First Search (DFS)
3. Greedy Best-First Search
4. A* Search
5. Hill Climbing
6. Map Coloring (Backtracking CSP)
7. Exit
Enter your choice (1-7): 1
Enter number of nodes: 6
Enter node names separated by spaces (example: A B C D):
1 2 3 4 5 6
Enter number of edges: 7
Enter each edge as: source destination cost
Example: A B 2
1 2 4
1 3 2
2 4 5
3 4 3
2 5 6
4 6 2
5 6 3
Enter start node: 1
Enter goal node: 6

Breadth-First Search (BFS) Result
Path: 1 -> 2 -> 4 -> 6
Path length (edges): 3
Nodes expanded: 6
Execution time: 0.028 ms

========== AI SEARCH ALGORITHMS ==========
1. Breadth-First Search (BFS)
2. Depth-First Search (DFS)
3. Greedy Best-First Search
4. A* Search
5. Hill Climbing
6. Map Coloring (Backtracking CSP)
7. Exit
Enter your choice (1-7): 2
Enter number of nodes: 6
Enter node names separated by spaces (example: A B C D):
1 2 3 4 5 6
Enter number of edges: 7
Enter each edge as: source destination